In [1]:
import polars as pl, numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")

In [2]:
df = (pl.read_parquet("data/interim/master_all_cities.parquet")
        .with_columns(pl.col("snapshot_date").str.to_date(strict=False).alias("snap")))

def arr(frame, col):
    return frame[col].drop_nulls().to_numpy()

def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

def rank_biserial(u, na, nb):        # effect size for Mann–Whitney
    return 1 - (2*u) / (na*nb)

def eta_epsilon(groups):             # effect sizes for ANOVA / Kruskal
    all_ = np.concatenate(groups); N = len(all_); k = len(groups)
    grand = all_.mean()
    ss_between = sum(len(g)*(g.mean()-grand)**2 for g in groups)
    ss_total = ((all_-grand)**2).sum()
    eta2 = ss_between/ss_total
    H, _ = stats.kruskal(*groups)
    eps2 = (H - k + 1) / (N - k)     # epsilon² from Kruskal H
    return eta2, eps2

def d_label(d):
    a=abs(d); return "negligible" if a<.2 else "small" if a<.5 else "medium" if a<.8 else "large"

# 5.1 Hypothesis Testing

### H1: Entre-home listings command statistically significantly higher prices than private rooms.

In [12]:
g = df.filter(pl.col("price").is_not_null())
entire  = arr(g.filter(pl.col("room_type")=="Entire home/apt"), "price")
private = arr(g.filter(pl.col("room_type")=="Private room"), "price")
le, lp = np.log(entire), np.log(private)
lev = stats.levene(le, lp).pvalue
t = stats.ttest_ind(le, lp, equal_var=False, alternative="greater")
u = stats.mannwhitneyu(entire, private, alternative="greater")
print(f"Levene p={lev:.2e} | Welch t={t.statistic:.1f}, p={t.pvalue:.2e} | MWU p={u.pvalue:.2e}")
print(f"Cohen's d(log)={cohen_d(le,lp):.2f} ({d_label(cohen_d(le,lp))}) | "
      f"median €{np.median(entire):.0f} vs €{np.median(private):.0f}")

Levene p=8.14e-21 | Welch t=109.5, p=0.00e+00 | MWU p=0.00e+00
Cohen's d(log)=1.55 (large) | median €317 vs €113


### H2 — Superhosts achieve higher review scores

In [13]:
s  = arr(df.filter(pl.col("host_is_superhost")==True),  "review_scores_rating")
ns = arr(df.filter(pl.col("host_is_superhost")==False), "review_scores_rating")
u = stats.mannwhitneyu(s, ns, alternative="greater")
print(f"MWU U={u.statistic:.0f}, p={u.pvalue:.2e}, rank-biserial={rank_biserial(u.statistic,len(s),len(ns)):.3f}")
print(f"median: super {np.median(s):.2f} vs non {np.median(ns):.2f}")

MWU U=153307825, p=0.00e+00, rank-biserial=-0.293
median: super 4.91 vs non 4.77


### H3 — Listings with >10 reviews price differently

In [14]:
d = df.filter(pl.col("price").is_not_null() & pl.col("number_of_reviews").is_not_null())
hi = arr(d.filter(pl.col("number_of_reviews")>10), "price")
lo = arr(d.filter(pl.col("number_of_reviews")<=10), "price")
t = stats.ttest_ind(np.log(hi), np.log(lo), equal_var=False)   # two-sided
u = stats.mannwhitneyu(hi, lo, alternative="two-sided")
print(f"Welch t={t.statistic:.1f}, p={t.pvalue:.2e} | MWU p={u.pvalue:.2e} | "
      f"d(log)={cohen_d(np.log(hi),np.log(lo)):.2f} ({d_label(cohen_d(np.log(hi),np.log(lo)))})")
print(f"median €{np.median(hi):.0f} vs €{np.median(lo):.0f}")

Welch t=-8.1, p=6.90e-16 | MWU p=1.87e-14 | d(log)=-0.10 (negligible)
median €272 vs €296


### H5: Weekend vs. weekday pricing differences are sta<s<cally significant

In [18]:
cal = con.execute("""
    SELECT cal.available,
           CASE WHEN isodow(d.date) IN (6,7) THEN 'weekend' ELSE 'weekday' END AS daytype
    FROM fact_calendar cal JOIN dim_date d ON d.date_key = cal.date_key
""").pl()
wk = cal.filter(pl.col("daytype")=="weekend")["available"].cast(pl.Int8).drop_nulls().to_numpy()
wd = cal.filter(pl.col("daytype")=="weekday")["available"].cast(pl.Int8).drop_nulls().to_numpy()
from statsmodels.stats.proportion import proportions_ztest
z,p = proportions_ztest([wk.sum(), wd.sum()], [len(wk), len(wd)])
h = 2*np.arcsin(np.sqrt(wk.mean())) - 2*np.arcsin(np.sqrt(wd.mean()))   # Cohen's h
print(f"available weekend {wk.mean():.3f} vs weekday {wd.mean():.3f} | z={z:.1f}, p={p:.2e}, Cohen's h={h:.3f}")

available weekend 0.468 vs weekday 0.471 | z=-8.1, p=5.71e-16, Cohen's h=-0.005
